# Open-weight scale ladder — judging run

Reproduces the judging step of the paper with open weights. Everything downstream of the labels
is deterministic, so this notebook is the only part that needs a GPU.

**Protocol is fixed in `scale_prereg.py` and this notebook does not change it.** Decoding is
temperature 0 / top_p 1 / max_tokens 1024 / seed 20260908, the strict parser is the only parser,
and Qwen3 thinking mode is disabled. Every run writes a manifest recording the resolved
HuggingFace commit sha, quantisation and serving-stack versions.

Runtime: **A100 or L4**. Set it under Runtime -> Change runtime type before running anything.

In [ ]:
# 1 · repository and dependencies
REPO = "https://github.com/Taekyoon/academic_ir_research_2026.git"

import os, subprocess, sys

# Clone, or reuse an existing checkout, and FAIL LOUDLY otherwise. An earlier version of this
# cell wrote `git clone ... 2>/dev/null || echo "already cloned"`, which reported success for a
# genuine failure and then broke confusingly on the next line.
if os.path.basename(os.getcwd()) == "work":
    print("already inside the checkout:", os.getcwd())
elif os.path.isdir("work/.git"):
    os.chdir("work")
    # Reusing a checkout must also UPDATE it, otherwise a re-run silently keeps whatever
    # commit was current when the session started and any fix pushed since is invisible.
    # reset --hard touches TRACKED files only, so a fetched clef_abstracts.jsonl and any
    # labels/ output produced in this session both survive.
    for cmd in (["git", "fetch", "--depth", "1", "origin", "main"],
                ["git", "reset", "--hard", "origin/main"]):
        rr = subprocess.run(cmd, capture_output=True, text=True)
        if rr.returncode != 0:
            raise SystemExit(" ".join(cmd) + " failed: " + rr.stderr.strip())
    print("reusing and updating existing checkout:", os.getcwd())
else:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO, "work"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(f"git clone failed ({r.returncode}):\n{r.stderr.strip()}")
    os.chdir("work"); print("cloned to:", os.getcwd())

for p in ("code/scale_judge.py", "data/panel_sample.csv", "prereg/scale_prereg.py"):
    assert os.path.exists(p), f"checkout is incomplete: {p} missing"
print("HEAD:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip())

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# BOTH versions are pinned, and transformers is the one that matters. vLLM 0.11.0 declares
# `transformers>=4.55.2` with NO upper bound, so pip leaves Colab preinstalled transformers 5.x
# in place - and transformers 5.x removed Tokenizer.all_special_tokens_extended, which vLLM
# 0.11.0 calls during tokenizer init. The run then dies with
#   AttributeError: Qwen2Tokenizer has no attribute all_special_tokens_extended
# after the model has already downloaded. vLLM 0.11.1 added `transformers<5` for this reason.
# Pinning transformers to the newest 4.x is the smallest fix; upgrading vLLM instead would pull
# a different pinned torch and force a multi-gigabyte reinstall.
!pip -q install "vllm==0.11.0" "transformers==4.57.6" "huggingface_hub>=0.26"

import vllm, torch, transformers
print("vllm", vllm.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__)
print("bf16 supported:", torch.cuda.is_bf16_supported())
assert transformers.__version__.startswith("4."), (
    "transformers " + transformers.__version__ + " is a 5.x release and vLLM 0.11.0 cannot use "
    "its tokenizer API. Re-run this cell, then Runtime -> Restart session, then run it again.")

## 2 · Abstracts

The repository ships PMIDs and expert labels, not abstract text — the CLEF collection itself
ships only PMIDs, and this keeps publisher-copyrighted abstracts out of the repository. Fetch
them once; the file is cached for the rest of the session.

`--email` is optional. NCBI asks automated callers to identify themselves; supply your own
address or omit the flag.

In [ ]:
!python code/fetch_abstracts.py --pmids data/pmids_panel.txt --out clef_abstracts.jsonl
!wc -l clef_abstracts.jsonl

# Expect about 2,017 records, of which roughly 7% carry a title and no abstract body. Those are
# judged on the title alone and the judging script flags them title_only.
import json
recs = [json.loads(l) for l in open("clef_abstracts.jsonl") if l.strip()]
empty = sum(1 for r in recs if not (r.get("abstract") or "").strip())
print(f"records {len(recs):,} | title-only {empty:,} ({empty/max(len(recs),1):.1%})")

## 2b · HuggingFace access for the gated families

Llama and Gemma are gated with **manual approval** on the HuggingFace API, not merely
token-gated. Two steps, both on your own account:

1. Visit each model page and accept the licence — `meta-llama/Llama-3.2-1B-Instruct`,
   `meta-llama/Llama-3.2-3B-Instruct`, `meta-llama/Llama-3.1-8B-Instruct`,
   `google/gemma-3-1b-it`, `google/gemma-3-4b-it`, `google/gemma-3-12b-it`.
   Approval is usually immediate but it is per model, not per family.
2. Put a **read** token in Colab Secrets under the name `HF_TOKEN` and enable it for this
   notebook (the key icon in the left sidebar).

The cell below reads the token from Colab Secrets so it never appears in the notebook and can
never be committed. If the secret is absent it prompts instead, with the input masked. It then
probes every gated repo and reports which are reachable — a family that is not reachable is
reported as **not attempted**, never substituted.

In [ ]:
import os, getpass
from huggingface_hub import login, HfApi

# Never write a token literal into this notebook. Colab Secrets keeps it out of the file and out
# of any commit; getpass is the fallback and masks the input.
tok = None
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    print("token source: Colab Secrets")
except Exception as e:
    print("Colab Secrets unavailable (" + type(e).__name__ + ")")
if not tok:
    tok = getpass.getpass("HF read token (input hidden, leave blank to skip gated families): ")
    print("token source: prompt")

GATED = ["meta-llama/Llama-3.2-1B-Instruct", "meta-llama/Llama-3.2-3B-Instruct",
         "meta-llama/Llama-3.1-8B-Instruct",
         "google/gemma-3-1b-it", "google/gemma-3-4b-it", "google/gemma-3-12b-it"]

if not tok:
    print("\nNo token supplied. The llama and gemma ladders will be NOT ATTEMPTED.")
    reachable = []
else:
    login(token=tok, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = tok          # vLLM and hub calls pick this up
    api = HfApi(token=tok)
    reachable, blocked = [], []
    for rid in GATED:
        try:
            api.model_info(rid)
            reachable.append(rid)
        except Exception as e:
            blocked.append((rid, type(e).__name__))
    print("\nreachable:")
    for r in reachable:
        print("   OK      " + r)
    for r, why in blocked:
        print("   BLOCKED " + r + "  (" + why + ")")
    if blocked:
        print("\nA BLOCKED repo means the licence has not been accepted by the account that owns")
        print("this token. Open the model page, accept, then re-run this cell. Do not work around")
        print("it with a mirror or an ungated copy - the pre-registration pins these repo ids.")

# never print or log the token itself
del tok

## 3 · Smoke test before spending the GPU

64 rows on the smallest arm. Check three things in the output: `strict` is 64/64, `out_tok med`
is small (a large median means a reasoning preamble is leaking through and thinking mode is not
actually off), and `rows/s` is high enough that the full run is affordable.

In [ ]:
import os
# The abstracts file is built by the cell above and is not shipped in the repository. Check it
# here so the smoke test cannot be run out of order.
assert os.path.exists("clef_abstracts.jsonl"), (
    "clef_abstracts.jsonl is missing - run the fetch cell above first")
n = sum(1 for _ in open("clef_abstracts.jsonl"))
print(f"abstracts on disk: {n:,} records")
assert n > 1900, f"only {n} records - the fetch looks incomplete, re-run the cell above"

!python code/scale_judge.py --arm qwen3-4b --condition A --limit 64 --outdir smoke
!cat smoke/scale_manifest_qwen3-4b_A.json

## 3 · The three ladders

One ladder per family, every size whose **bf16** weights fit the device. Precision is bf16 for
every arm and there is no quantisation option, so the reachable ladder is a property of the card:

| family | A100 40 GB | adds on 80 GB | span |
|---|---|---|---|
| Qwen3 | 1.7B · 4B · 8B · 14B | 32B | 7.3x (16.1x) |
| Llama | 1B · 3B · 8B | — (70B is 131 GB) | 6.5x |
| Gemma | 1B · 4B · 12B | 27B | 12.2x (27.4x) |

An arm that does not fit is refused **before the model loads** and reported as not attempted.
The cell prints what this session can actually reach before it starts, and skips a family whose
gated access was not granted above.

In [ ]:
import subprocess, torch, sys
sys.path.insert(0, "code")
from scale_judge import ARMS, WEIGHT_GB, PARAMS

gb = torch.cuda.get_device_properties(0).total_memory / 2**30
cap = gb * 0.90
FAM = {"qwen3": ["qwen3-1.7b", "qwen3-4b", "qwen3-8b", "qwen3-14b", "qwen3-32b"],
       "llama": ["llama-3.2-1b-instruct", "llama-3.2-3b-instruct", "llama-3.1-8b-instruct"],
       "gemma": ["gemma-3-1b-it", "gemma-3-4b-it", "gemma-3-12b-it", "gemma-3-27b-it"]}

granted = {r.split("/")[-1].lower() for r in reachable} if "reachable" in dir() else set()
def ok(arm):
    if WEIGHT_GB[arm] > cap:
        return False, f"bf16 {WEIGHT_GB[arm]} GB exceeds {cap:.0f} GB usable"
    if not arm.startswith("qwen3") and ARMS[arm].split("/")[-1].lower() not in granted:
        return False, "gated access not granted"
    return True, ""

print(f"{torch.cuda.get_device_name(0)} | {gb:.0f} GB | usable {cap:.0f} GB\n")
plan = {}
for fam, arms in FAM.items():
    run = [a for a in arms if ok(a)[0]]
    plan[fam] = run
    span = max(PARAMS[a] for a in run) / min(PARAMS[a] for a in run) if len(run) > 1 else 1.0
    print(f"{fam:6s} {len(run)} points, span {span:.1f}x  {run}")
    for a in arms:
        good, why = ok(a)
        if not good:
            print(f"         NOT ATTEMPTED  {a}  ({why})")
total = sum(len(v) for v in plan.values()) * 2
print(f"\n{total} arm-conditions x 2,025 pairs = {total*2025:,} judgements")

for fam, arms in plan.items():
    for arm in arms:
        for cond in ("A", "C"):
            print(f"\n===== {fam} / {arm} / condition {cond}", flush=True)
            r = subprocess.run(["python", "code/scale_judge.py", "--arm", arm,
                                "--condition", cond])
            if r.returncode != 0:
                print(f"  {arm} {cond} did not complete - reported as not attempted, "
                      f"NOT quantised and NOT substituted", flush=True)

## 6 · Collect

Zip the labels and manifests and download. The analysis runs off these two file types alone and
needs no GPU.

**If any arm printed a strict-parse rate below 0.95**, do not analyse it yet — the
pre-registration (R2) requires re-running that arm with `--guided` AND running one arm that
passed with `--guided` too, so the harness effect can be bounded. Free and guided labels are
never mixed within an arm.

In [ ]:
!zip -qr scale_labels.zip labels/
!ls -la labels/ | head -30
from google.colab import files
files.download("scale_labels.zip")